# Oppitunti 13 - Agentin muisti


## Setup

This notebook demonstrates how to build a travel booking agent with **persistent memory** using the **Microsoft Agent Framework** (MAF).

You will learn how different types of agent memory — working, short-term, and long-term — affect how an agent retains and uses information across conversations.

**Prerequisites:**
- A Microsoft Foundry project with a deployed chat model (e.g. `gpt-5-mini`).
- Logged in with the Azure CLI — run `az login` in your terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — your Microsoft Foundry project endpoint.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Agentin muistityypit

AI-agentit voivat hyödyntää erilaisia muistityyppejä, joilla kullakin on oma tarkoituksensa:

### Työmuisti
Itse keskusteluketju — yhdessä istunnossa vaihdetut viestit. Agentti voi viitata aiempiin viesteihin samassa ketjussa ylläpitääkseen johdonmukaisuutta. MAF:ssa tämä luodaan **`agent.create_session()`** -komennolla, joka palauttaa `AgentSession`-objektin.

### Lyhytkestoinen muisti
Tiedot, jotka säilyvät tehtävän tai istunnon ajan, mutta joita ei tallenneta pysyvästi. Esimerkiksi agentti voi kerätä faktoja monivaiheisen suunnittelukeskustelun aikana ja käyttää niitä lopullisen matkasuunnitelman laatimiseen.

### Pitkäkestoinen muisti
Mieltymykset ja faktat, jotka säilyvät **istuntojen välillä**. Palaavan käyttäjän ei pitäisi joutua toistamaan dieetinarajoituksiaan tai matkustustyyliään. Pitkäkestoinen muisti on tyypillisesti tuettu ulkoisella tallennusratkaisulla — tietokanta, tiedosto tai vektori-indeksi — ja se tuodaan agentin käyttöön työkaluilla.


## Työmuisti istuntojen kanssa

Muistin yksinkertaisin muoto on keskusteluistunto. Kun annat saman istunnon (luotu `agent.create_session()` -komennolla) peräkkäisille `agent.run()` -kutsuille, agentti näkee koko keskusteluhistorian ja voi muistaa aiemmat yksityiskohdat.

Luodaan matkatoimistoagentti ja demonstroidaan työmuistia.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agentti muisti budjetin oikein, koska molemmat viestit jakavat saman istunnon. Tämä on **työmuisti** — se on olemassa vain istunnon elinkaaren ajan.

### Mitä tapahtuu uudella ketjulla?

Jos luomme **uuden** istunnon, agentilla ei ole muistikuvaa edellisestä keskustelusta:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Pitkän aikavälin muistikuviot

Käyttäjäasetusten muistamiseksi **istuntojen välillä** tarvitsemme pysyvän tallennustilan, joka on keskusteluketjun ulkopuolella. Agentti käyttää tätä tallennustilaa **työkalujen** kautta — funktioita, joita se voi kutsua tietojen tallentamiseen ja hakemiseen.

Alla toteutamme yksinkertaisen muistissa pidettävän asetustallennustilan (tuotannossa tämän taustalla olisi tietokanta tai vektori-indeksi) ja teemme siitä työkaluja, joita agentti voi käyttää.

### Arkkitehtuuri
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Tapaus 1 — Ensikertalainen varaa vuosipäivämatkan

Sarah vierailee ensimmäistä kertaa. Asiakkaan tulisi tallentaa hänen mieltymyksensä työkalujen avulla ja käyttää niitä hotellien suosittelussa.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Tapaus 2 — Sarah palaa viikkoja myöhemmin

Sarah aloittaa **täysin uuden keskustelun** (simuloiden uutta istuntoa). Työmuisti on tyhjä, mutta pitkäaikaisessa asetusten tallennuspaikassa on yhä hänen tietonsa. Agentin tulisi hakea ne ja käyttää niitä räätälöityjen suositusten antamiseen.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Yhteenveto

Tässä oppitunnissa opit kolme agentin muistityyppiä ja miten toteuttaa ne Microsoft Agent Frameworkilla:

| Muistityyppi | MAF-mekanismi | Kesto |
|---|---|---|
| **Työmuisti** | `agent.create_session()` | Yksi keskustelu |
| **Lyhytkestoinen** | Kertynyt konteksti säikeen sisällä | Yksi tehtävä / istunto |
| **Pitkäkestoinen** | Ulkoinen tallennus, johon pääsee `@tool`-funktioilla | Istuntojen yli |

### Tärkeimmät opit
1. **`agent.create_session()`** tarjoaa työmuistin — agentti näkee koko keskusteluhistorian istunnon aikana.
2. **Uudet istunnot menettävät kontekstin** — ilman pitkäkestoista muistia agentti ei muista aiempia keskusteluja.
3. **`@tool`-funktiot** ylittävät kuilun — ne antavat agentille mahdollisuuden tallentaa ja hakea tietoa pysyvästä tallennuspaikasta.
4. **Personalisointi paranee ajan myötä** — mitä enemmän mieltymyksiä tallennetaan, sitä parempia suosituksia agentti pystyy antamaan.

### Käytännön sovellukset
- **Asiakaspalvelu**: Muistaa asiakkaan historian ja mieltymykset
- **Henkilökohtaiset avustajat**: Säilyttää kontekstin päivien tai viikkojen ajan
- **Terveydenhuolto**: Seuraa potilastietoja ja mieltymyksiä
- **Verkkokauppa**: Personoitu ostokokemus historian perusteella

### Seuraavat askeleet
- Korvaa muistissa oleva sanakirja tietokannalla tai vektorisäilöllä (esim. Azure AI Search)
- Lisää muistin vanhentuminen ajallisesti herkkää tietoa varten
- Rakenna monen agentin järjestelmiä, joilla on jaettu muisti
- Tutustu Cognee-muistikirjaan, joka perustuu tietämyspohjaiseen muistiin


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
